In [1]:
#(1)_インポート
import os
import csv
import re
import time
import random
import glob
import logging
import io
import contextlib
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

import pandas as pd

try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance が見つかりません。pip install yfinance を実行してください。")

# yfinance のログを抑制（完全には消えない場合もあります）
logging.getLogger("yfinance").setLevel(logging.ERROR)


#(2)_設定（★更新：モード別プロファイル＋statusパス追加）
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")
OUT_DIR = os.path.join("Data", "kabuka")

# ★ status保存（上場廃止などの状態管理）
STATUS_CSV = os.path.join("Data", "kabuka", "status.csv")

# =========================
# モード別プロファイル
# =========================
# reliable: 遅くてOK、欠損を潰す
RELIABLE = dict(
    BATCH_SIZE=200,
    MIN_BATCH_SIZE=50,
    SAVE_EVERY=400,          # checkpoint頻度（銘柄数基準）
    PRINT_EVERY=400,
    SLEEP_PER_BATCH=1.0,
    JITTER_SEC=0.8,
    YF_TIMEOUT_SEC=240.0,
    YF_THREADS=False,        # 安定優先
    COVERAGE_MIN_RATIO=0.85,
    RETRY_PASSES=3,
    RETRY_BATCH_SIZE=50,
    RETRY_SINGLE_MAX=400,
)

# fast: 直近更新を速く
FAST = dict(
    BATCH_SIZE=1200,
    MIN_BATCH_SIZE=200,
    SAVE_EVERY=3000,
    PRINT_EVERY=3000,
    SLEEP_PER_BATCH=0.0,
    JITTER_SEC=0.2,
    YF_TIMEOUT_SEC=120.0,
    YF_THREADS=True,         # 環境でハングするならFalseに
    COVERAGE_MIN_RATIO=0.60, # 短期更新は厳しすぎない
    RETRY_PASSES=1,
    RETRY_BATCH_SIZE=200,
    RETRY_SINGLE_MAX=200,
)

RECENT_DAYS_IF_EMPTY = 7

# ★ 上場廃止判定（1か月=31日を採用）
DELIST_DAYS = 31

# ※ profile（RELIABLE/FAST）を基本に使うが、関数のデフォルト引数参照のため最低限定義しておく
YF_TIMEOUT_SEC = 180.0
YF_THREADS = False
COVERAGE_MIN_RATIO = 0.85

#(3-1)_銘柄コード→yfinance用ticker変換（東証: 7203 -> 7203.T / 日経平均: N225 -> ^N225）
def code_to_ticker(code: str) -> str:
    code = str(code).strip()
    if not code:
        return ""

    # ★日経平均（Nikkei 225）
    if code.upper() == "N225":
        return "^N225"

    # 東証（例：7203 -> 7203.T）
    return f"{code}.T"

#(3-2)_年度CSVの読み込み（指定フォーマット）
def load_year_csv(path: str):
    """
    期待フォーマット:
      1行目: ["code", <code1>, <code2>, ...]
      2行目: ["name", <name1>, <name2>, ...]
      3行目以降: ["YYYY-MM-DD", v1, v2, ...]
    戻り値:
      (names: dict[code]=name, df: index=Timestamp(毎日), columns=code(str), values=float)
      ファイルが無ければ ({} , None)
    """
    if not os.path.exists(path):
        return {}, None

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.reader(f))

    if len(rows) < 2:
        return {}, None

    header_codes = rows[0]
    header_names = rows[1]

    if not header_codes or header_codes[0] != "code":
        raise ValueError(f"想定外フォーマット（1行目先頭が 'code' ではありません）: {path}")
    if not header_names or header_names[0] != "name":
        raise ValueError(f"想定外フォーマット（2行目先頭が 'name' ではありません）: {path}")

    codes = [c.strip() for c in header_codes[1:] if c.strip()]

    # name 行は codes に合わせて取り出し（長さが足りない場合は空）
    names = {}
    for i, c in enumerate(codes):
        nm = header_names[i + 1].strip() if i + 1 < len(header_names) else ""
        names[c] = nm

    # データ行
    data_rows = rows[2:]
    if not data_rows:
        df = pd.DataFrame(columns=codes)
        df.index.name = "date"
        return names, df

    dates = []
    values = []
    for r in data_rows:
        if not r:
            continue
        d = r[0].strip()
        if not d:
            continue
        try:
            dt = pd.to_datetime(d)
        except Exception:
            continue
        dates.append(dt)

        row_vals = []
        for j in range(len(codes)):
            cell = r[j + 1].strip() if j + 1 < len(r) else ""
            if cell == "":
                row_vals.append(pd.NA)
            else:
                try:
                    row_vals.append(float(cell))
                except Exception:
                    row_vals.append(pd.NA)
        values.append(row_vals)

    df = pd.DataFrame(values, index=pd.DatetimeIndex(dates), columns=codes)
    df = df.sort_index()
    df.index.name = "date"
    return names, df

#(3-3)_年度CSVの保存（指定フォーマット / code昇順で保存）（★更新：N225を最後尾に固定）
def save_year_csv(path: str, names: dict, df: pd.DataFrame):
    """
    df: index=DatetimeIndex(日次), columns=code(str)
    names: dict[code]=name
    """
    Path(os.path.dirname(path)).mkdir(parents=True, exist_ok=True)

    # columns を code 昇順（数値優先）で並べ替え
    # ★ N225 は必ず最後尾に固定
    def code_key(c):
        c = str(c)
        if c.upper() == "N225":
            return (2, 10**18)  # 最後尾固定
        try:
            return (0, int(c))  # 数値コードは先頭側
        except Exception:
            return (1, c)       # その他文字列は中間

    codes_sorted = sorted([str(c) for c in df.columns], key=code_key)
    df2 = df.reindex(columns=codes_sorted)

    # names 行も合わせる（無ければ空）
    name_row = ["name"] + [names.get(c, "") for c in codes_sorted]
    code_row = ["code"] + codes_sorted

    # 日付はYYYY-MM-DDで出力
    out_rows = [code_row, name_row]
    for dt, row in df2.iterrows():
        dstr = pd.to_datetime(dt).strftime("%Y-%m-%d")
        vals = []
        for c in codes_sorted:
            v = row[c]
            if pd.isna(v):
                vals.append("")
            else:
                vals.append(str(float(v)))
        out_rows.append([dstr] + vals)

    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerows(out_rows)

#(3-4)_01_IDmap.csv から銘柄コード一覧を取得（code列）
def load_codes_from_idmap(idmap_csv: str):
    df = pd.read_csv(idmap_csv, dtype=str)
    if "code" not in df.columns:
        raise ValueError("01_IDmap.csv に 'code' カラムが見つかりません。")

    codes = (
        df["code"]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != ""]
        .unique()
        .tolist()
    )
    return codes

#(3-5)_yfinanceで日足終値（Close）を取得（バッチ対応 / タイムアウト対応）（★更新：ticker→codeで列名を確実に戻す）
def fetch_close_df_yf_batch(
    codes: list,
    start: str,
    end: str,
    timeout_sec: float = YF_TIMEOUT_SEC,
    threads: bool = YF_THREADS,
    suppress_yf_stdout: bool = True,
) -> pd.DataFrame:
    """
    codes: ["1301", "1322", "N225", ...]
    start/end: 'YYYY-MM-DD'（endは排他的）
    戻り: index=DatetimeIndex, columns=code(str), values=Close(float)
    """
    codes = [str(c).strip() for c in codes if str(c).strip() != ""]
    if not codes:
        return pd.DataFrame()

    # code -> ticker
    tickers = [code_to_ticker(c) for c in codes]

    # ★ ticker -> code（^N225 を N225 に戻すため必須）
    ticker_to_code = {code_to_ticker(c): str(c).strip() for c in codes}

    def _do_download():
        if suppress_yf_stdout:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                return yf.download(
                    tickers,
                    start=start,
                    end=end,
                    interval="1d",
                    auto_adjust=False,
                    progress=False,
                    group_by="column",
                    threads=threads,
                )
        else:
            return yf.download(
                tickers,
                start=start,
                end=end,
                interval="1d",
                auto_adjust=False,
                progress=False,
                group_by="column",
                threads=threads,
            )

    # タイムアウト付きで実行
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(_do_download)
            hist = fut.result(timeout=timeout_sec)
    except FuturesTimeoutError:
        print(f"  [TIMEOUT] yf.download timeout {timeout_sec}s (tickers={len(codes)})", flush=True)
        return pd.DataFrame()
    except Exception as e:
        print(f"  [ERROR] yf.download failed: {type(e).__name__}: {e}", flush=True)
        return pd.DataFrame()

    if hist is None or hist.empty:
        return pd.DataFrame()

    close_df = None

    if isinstance(hist.columns, pd.MultiIndex):
        # 一般的：1階層目に "Close"
        try:
            if "Close" in hist.columns.get_level_values(0):
                close_df = hist["Close"]
        except Exception:
            close_df = None

        # 別形：2階層目に "Close"
        if close_df is None:
            try:
                lvl1 = hist.columns.get_level_values(1)
                if any(str(x).lower() == "close" for x in lvl1):
                    cols = [c for c in hist.columns if str(c[1]).lower() == "close"]
                    tmp = hist[cols].copy()
                    tmp.columns = [c[0] for c in cols]  # ticker名へ
                    close_df = tmp
            except Exception:
                close_df = None
    else:
        # 単一tickerの形
        if "Close" in hist.columns and len(tickers) == 1:
            close_df = hist[["Close"]].copy()
            close_df.columns = [tickers[0]]

    if close_df is None or close_df.empty:
        return pd.DataFrame()

    # ★ columns: ticker -> code（^N225 -> N225 / 1301.T -> 1301）
    col_map = {}
    for t in list(close_df.columns):
        tt = str(t)
        if tt in ticker_to_code:
            col_map[t] = ticker_to_code[tt]
        elif tt.endswith(".T"):
            col_map[t] = tt[:-2]
        else:
            col_map[t] = tt

    close_df = close_df.rename(columns=col_map)

    close_df.index = pd.to_datetime(close_df.index)
    close_df = close_df.apply(pd.to_numeric, errors="coerce")
    return close_df

#(3-6)_status.csv の読み書き＆上場廃止判定（★更新：active-onlyフィルタ追加）
def load_status_csv(path: str) -> pd.DataFrame:
    """
    status.csv schema:
      code(str), last_ok_date(YYYY-MM-DD or blank), last_try_date(YYYY-MM-DD or blank),
      consecutive_empty(int), status(active/delisted/suspect)
    """
    if not os.path.exists(path):
        return pd.DataFrame(columns=["code", "last_ok_date", "last_try_date", "consecutive_empty", "status"])

    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    for c in ["code", "last_ok_date", "last_try_date", "consecutive_empty", "status"]:
        if c not in df.columns:
            df[c] = ""

    df["code"] = df["code"].astype(str).str.strip()
    df = df.loc[df["code"] != ""].copy()

    # 型寄せ
    df["consecutive_empty"] = pd.to_numeric(df["consecutive_empty"], errors="coerce").fillna(0).astype(int)
    df["status"] = df["status"].fillna("").astype(str)
    df["last_ok_date"] = df["last_ok_date"].fillna("").astype(str)
    df["last_try_date"] = df["last_try_date"].fillna("").astype(str)

    df = df.drop_duplicates(subset=["code"], keep="last").reset_index(drop=True)
    return df


def filter_codes_by_status(codes: list[str], status_df: pd.DataFrame, include_delisted: bool = False) -> list[str]:
    """
    include_delisted=False のとき、status=delisted は除外
    """
    codes = [str(c).strip() for c in codes if str(c).strip() != ""]
    if status_df is None or status_df.empty:
        return codes

    st = status_df.copy()
    st["code"] = st["code"].astype(str).str.strip()
    st["status"] = st["status"].fillna("").astype(str).str.strip()
    st_map = dict(zip(st["code"], st["status"]))

    out = []
    for c in codes:
        s = st_map.get(c, "")
        if (not include_delisted) and (s == "delisted"):
            continue
        out.append(c)
    return out


def filter_codes_active_only(codes: list[str], status_df: pd.DataFrame) -> list[str]:
    """
    active-only:
      - last_ok_date が空でない銘柄のみ残す（直近で一度でも取れた銘柄だけ遡る）
    """
    codes = [str(c).strip() for c in codes if str(c).strip() != ""]
    if status_df is None or status_df.empty:
        return []

    st = status_df.copy()
    st["code"] = st["code"].astype(str).str.strip()
    st["last_ok_date"] = st["last_ok_date"].fillna("").astype(str).str.strip()

    ok_set = set(st.loc[st["last_ok_date"] != "", "code"].tolist())
    return [c for c in codes if c in ok_set]


def save_status_csv(path: str, df: pd.DataFrame):
    Path(os.path.dirname(path)).mkdir(parents=True, exist_ok=True)
    out = df.copy()

    # 並び：code昇順（数値優先）
    def code_key(c):
        try:
            return int(str(c))
        except Exception:
            return 10**18

    out["code"] = out["code"].astype(str)
    out = out.sort_values(by="code", key=lambda s: s.map(code_key)).reset_index(drop=True)

    out.to_csv(path, index=False, encoding="utf-8-sig")

#(3-6-0)_status判定ユーティリティ（★追加：日付・delisted判定）
def _today_jst_date_str() -> str:
    return pd.Timestamp.now(tz="Asia/Tokyo").normalize().strftime("%Y-%m-%d")


def _parse_date_or_none(s: str):
    s = (s or "").strip()
    if not s:
        return None
    try:
        return pd.to_datetime(s).normalize()
    except Exception:
        return None


def mark_delisted_by_rule(status_df: pd.DataFrame, delist_days: int = DELIST_DAYS) -> pd.DataFrame:
    """
    ルール：
      - last_ok_date から delist_days 超過なら delisted
      - last_ok_date が空なら suspect（判定保留）
      - last_ok_date が新しければ active
    """
    today = pd.to_datetime(_today_jst_date_str())
    out = status_df.copy()

    if out is None or out.empty:
        return out

    # 必須列を保証
    for c in ["code", "last_ok_date", "last_try_date", "consecutive_empty", "status"]:
        if c not in out.columns:
            out[c] = ""

    out["code"] = out["code"].astype(str).str.strip()
    out["status"] = out["status"].fillna("").astype(str).str.strip()
    out["last_ok_date"] = out["last_ok_date"].fillna("").astype(str)

    last_ok = out["last_ok_date"].map(_parse_date_or_none)
    days_since_ok = last_ok.map(lambda d: (today - d).days if d is not None else None)

    new_status = []
    for i, row in out.iterrows():
        d = days_since_ok.iloc[i]
        cur = (row.get("status") or "").strip()

        if d is None:
            # last_ok_date 不明は delisted にしない（疑い扱い）
            new_status.append("suspect" if cur == "" else cur)
        else:
            if d > delist_days:
                new_status.append("delisted")
            else:
                new_status.append("active")

    out["status"] = new_status
    return out


#(3-6-2)_status更新（取得結果をstatus.csvへ反映）（★更新：関数が途中で切れているため全文差し替え）
def update_status_after_fetch(
    status_df: pd.DataFrame,
    codes: list[str],
    close_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    close_df: index=取引日, columns=code, values=close
    - last_try_date: 今日
    - last_ok_date: close_df でその銘柄に非NAがあれば最大日付へ更新
    - consecutive_empty: 今回close_dfで非NAが無ければ+1、あれば0（目安）
    - 最後に delist ルールで status を更新
    """
    today_str = _today_jst_date_str()

    if status_df is None or status_df.empty:
        status_df = pd.DataFrame(columns=["code", "last_ok_date", "last_try_date", "consecutive_empty", "status"])

    st = status_df.copy()
    for col in ["code", "last_ok_date", "last_try_date", "consecutive_empty", "status"]:
        if col not in st.columns:
            st[col] = ""

    st["code"] = st["code"].astype(str).str.strip()
    st = st.loc[st["code"] != ""].copy()
    st = st.drop_duplicates(subset=["code"], keep="last").reset_index(drop=True)

    st["consecutive_empty"] = pd.to_numeric(st["consecutive_empty"], errors="coerce").fillna(0).astype(int)
    st["status"] = st["status"].fillna("").astype(str)
    st["last_ok_date"] = st["last_ok_date"].fillna("").astype(str)
    st["last_try_date"] = st["last_try_date"].fillna("").astype(str)

    st_map = {row["code"]: i for i, row in st.iterrows()}

    # close_df を正規化（念のため）
    if close_df is None or (isinstance(close_df, pd.DataFrame) and close_df.empty):
        close_df_norm = pd.DataFrame()
    else:
        close_df_norm = close_df.copy()
        close_df_norm.index = pd.to_datetime(close_df_norm.index).normalize()

    for c in codes:
        c = str(c).strip()
        if not c:
            continue

        if c not in st_map:
            st.loc[len(st)] = dict(
                code=c,
                last_ok_date="",
                last_try_date="",
                consecutive_empty=0,
                status="suspect",
            )
            st_map[c] = len(st) - 1

        i = st_map[c]
        st.at[i, "last_try_date"] = today_str

        got_any = False
        last_ok = None

        if isinstance(close_df_norm, pd.DataFrame) and (not close_df_norm.empty) and (c in close_df_norm.columns):
            s = close_df_norm[c]
            s2 = s.dropna()
            if not s2.empty:
                got_any = True
                last_ok = pd.to_datetime(s2.index.max()).normalize()

        if got_any and (last_ok is not None):
            st.at[i, "last_ok_date"] = last_ok.strftime("%Y-%m-%d")
            st.at[i, "consecutive_empty"] = 0
            # 取得できたら active に寄せる（ただし最終判定は mark_delisted_by_rule に委ねる）
            st.at[i, "status"] = "active"
        else:
            prev = int(pd.to_numeric(st.at[i, "consecutive_empty"], errors="coerce") or 0)
            st.at[i, "consecutive_empty"] = prev + 1
            # 取れない状態が続くものは suspect に寄せる
            cur = (st.at[i, "status"] or "").strip()
            if cur == "":
                st.at[i, "status"] = "suspect"

    # 最終的な delisted 判定（last_ok_dateから31日超ならdelisted）
    st = mark_delisted_by_rule(st, delist_days=DELIST_DAYS)

    st["consecutive_empty"] = pd.to_numeric(st["consecutive_empty"], errors="coerce").fillna(0).astype(int)
    st["code"] = st["code"].astype(str).str.strip()
    st = st.loc[st["code"] != ""].copy()
    st = st.drop_duplicates(subset=["code"], keep="last").reset_index(drop=True)
    return st

#(4-1)_年度ごとに「取引日だけ」株価を更新して Data/kabuka/YYYY.csv に保存（★更新：active-only warmup対応 + N225対応）
def _update_kabuka_core(
    start_date: str,
    end_date: str,
    profile: dict,
    include_delisted: bool,
    label: str,
    active_only: bool = False,
    warmup_days: int = 120,
):
    """
    共通エンジン：
      - 保存も更新も「取引日だけ」
      - status.csv を更新
      - profile により reliable / fast を切替

    active_only=True の場合：
      1) まず直近 warmup_days 日（暦日）を FAST で全銘柄に取りに行って status を作る
      2) last_ok_date が入った銘柄（＝active）だけを対象に start_date~end_date を回す
    """
    # プロファイル反映
    BATCH_SIZE = int(profile["BATCH_SIZE"])
    MIN_BATCH_SIZE = int(profile["MIN_BATCH_SIZE"])
    SAVE_EVERY = int(profile["SAVE_EVERY"])
    PRINT_EVERY = int(profile["PRINT_EVERY"])
    SLEEP_PER_BATCH = float(profile["SLEEP_PER_BATCH"])
    JITTER_SEC = float(profile["JITTER_SEC"])
    YF_TIMEOUT_SEC_LOCAL = float(profile["YF_TIMEOUT_SEC"])
    YF_THREADS_LOCAL = bool(profile["YF_THREADS"])
    COVERAGE_MIN_RATIO_LOCAL = float(profile["COVERAGE_MIN_RATIO"])
    RETRY_PASSES_LOCAL = int(profile["RETRY_PASSES"])
    RETRY_BATCH_SIZE_LOCAL = int(profile["RETRY_BATCH_SIZE"])
    RETRY_SINGLE_MAX_LOCAL = int(profile["RETRY_SINGLE_MAX"])

    codes = load_codes_from_idmap(IDMAP_CSV)

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    if end_ts < start_ts:
        raise ValueError("end_date は start_date 以降にしてください。")

    # years は start_ts〜end_ts の範囲で確定
    years = list(range(start_ts.year, end_ts.year + 1))

    # コード検証（4桁 or 3桁+英字 or 4桁+英字 + N225）
    pat_ok = re.compile(r"^(?:\d{4}|\d{3}[A-Z]|\d{4}[A-Z]|N225)$", re.IGNORECASE)

    codes_clean_all = [str(c).strip() for c in codes if str(c).strip() != ""]

    # ★ N225 を必ず対象に追加
    if "N225" not in {c.upper() for c in codes_clean_all}:
        codes_clean_all.append("N225")

    codes_valid_all = [c for c in codes_clean_all if pat_ok.match(c)]
    codes_invalid = [c for c in codes_clean_all if not pat_ok.match(c)]
    if codes_invalid:
        print(f"[SKIP] invalid code format (not fetched): {codes_invalid}", flush=True)

    # status 読み込み＆判定更新
    status_df = load_status_csv(STATUS_CSV)
    status_df = mark_delisted_by_rule(status_df, delist_days=DELIST_DAYS)

    # まず delisted を除外/含むを適用
    codes_valid = filter_codes_by_status(codes_valid_all, status_df, include_delisted=include_delisted)

    # =========================
    # active-only warmup
    # =========================
    if active_only:
        today = pd.Timestamp.now(tz="Asia/Tokyo").normalize().tz_localize(None)
        warmup_start = (today - pd.Timedelta(days=int(warmup_days))).strftime("%Y-%m-%d")
        warmup_end = today.strftime("%Y-%m-%d")

        print(f"[WARMUP] active-only: update recent {warmup_days} days (calendar): {warmup_start} - {warmup_end}", flush=True)

        # warmupはFASTプロファイルで回す
        _update_kabuka_core(
            warmup_start,
            warmup_end,
            FAST,
            include_delisted=False,
            label="WARMUP",
            active_only=False,
            warmup_days=0,
        )

        # warmup後のstatusを読み直し、activeのみ抽出
        status_df = load_status_csv(STATUS_CSV)
        status_df = mark_delisted_by_rule(status_df, delist_days=DELIST_DAYS)

        codes_valid = filter_codes_active_only(codes_valid_all, status_df)
        print(f"[ACTIVE-ONLY] filtered codes: {len(codes_valid)} / {len(codes_valid_all)} (by last_ok_date)", flush=True)

        if not codes_valid:
            print("[ACTIVE-ONLY] no active codes found after warmup. stop.", flush=True)
            return

        include_delisted = False

    print(f"[START:{label}] {start_date} - {end_date} years={years} codes(valid)={len(codes_valid)} skipped={len(codes_invalid)} include_delisted={include_delisted} active_only={active_only}", flush=True)

    def _chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    # 年ごと
    for year in years:
        year_start = max(start_ts, pd.Timestamp(year=year, month=1, day=1))
        year_end = min(end_ts, pd.Timestamp(year=year, month=12, day=31))

        # yfinance の end は排他的
        yf_start = year_start.strftime("%Y-%m-%d")
        yf_end_year = (year_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

        out_path = os.path.join(OUT_DIR, f"{year}.csv")

        names_existing, df_existing = load_year_csv(out_path)

        if df_existing is None:
            df_existing = pd.DataFrame()
            df_existing.index = pd.DatetimeIndex([], name="date")
            names = {}
        else:
            names = dict(names_existing)
            if not df_existing.empty:
                mask_any = df_existing.notna().any(axis=1)
                df_existing = df_existing.loc[mask_any].copy()
                df_existing.index = pd.to_datetime(df_existing.index).normalize()
                df_existing = df_existing[~df_existing.index.duplicated(keep="last")].sort_index()

        # ★ name 行（任意）
        if "N225" not in names:
            names["N225"] = "Nikkei 225"

        # 列追加（まとめて）
        existing_cols = set(map(str, df_existing.columns))
        missing_codes = [c for c in codes_valid if c not in existing_cols]
        if missing_codes:
            add_df = pd.DataFrame(pd.NA, index=df_existing.index, columns=missing_codes)
            df_existing = pd.concat([df_existing, add_df], axis=1)
        df_existing = df_existing.copy()

        total = len(codes_valid)
        print(f"[YEAR:{label}] {year} range: {year_start.date()} - {year_end.date()} target_codes={total}", flush=True)

        processed = 0

        for start_i in range(0, total, BATCH_SIZE):
            t0 = time.time()

            batch_codes = codes_valid[start_i:start_i + BATCH_SIZE]
            end_i = start_i + len(batch_codes)

            if start_i == 0 or (end_i % PRINT_EVERY == 0) or (end_i >= total):
                print(f"  ... batch {start_i+1}-{end_i}/{total} (size={len(batch_codes)}) start={yf_start} end={yf_end_year}", flush=True)

            def _fetch_batch_local(codes_sub):
                return fetch_close_df_yf_batch(
                    codes_sub, yf_start, yf_end_year,
                    timeout_sec=YF_TIMEOUT_SEC_LOCAL,
                    threads=YF_THREADS_LOCAL,
                    suppress_yf_stdout=True,
                )

            def _fetch_split_safe_local(codes_sub, init_batch_size, min_batch_size, coverage_min_ratio):
                codes_sub = [str(c).strip() for c in codes_sub if str(c).strip() != ""]
                if not codes_sub:
                    return pd.DataFrame()

                out_list = []
                i2 = 0
                cur_init = init_batch_size
                while i2 < len(codes_sub):
                    size = min(cur_init, len(codes_sub) - i2)
                    chunk = codes_sub[i2:i2 + size]

                    df = _fetch_batch_local(chunk)

                    ratio = (len(df.columns) / len(chunk)) if (len(chunk) > 0 and isinstance(df, pd.DataFrame)) else 0.0
                    if df.empty or ratio < coverage_min_ratio:
                        if size <= min_batch_size:
                            print(f"  [UNSTABLE] size={size} ratio={ratio:.2f} -> give up split here", flush=True)
                            i2 += size
                            continue
                        new_size = max(min_batch_size, size // 2)
                        print(f"  [SPLIT] unstable batch size={size} ratio={ratio:.2f} -> split_size={new_size}", flush=True)
                        cur_init = new_size
                        continue

                    out_list.append(df)
                    i2 += size

                if not out_list:
                    return pd.DataFrame()
                return pd.concat(out_list, axis=1)

            close_df = _fetch_split_safe_local(
                batch_codes,
                init_batch_size=len(batch_codes),
                min_batch_size=MIN_BATCH_SIZE,
                coverage_min_ratio=COVERAGE_MIN_RATIO_LOCAL,
            )

            got_cols = set(map(str, close_df.columns)) if isinstance(close_df, pd.DataFrame) else set()
            still_missing = [c for c in batch_codes if c not in got_cols]

            if still_missing:
                print(f"  [MISS] initial missing={len(still_missing)}/{len(batch_codes)} -> retry", flush=True)

                for p in range(RETRY_PASSES_LOCAL):
                    if not still_missing:
                        break

                    retried_list = []
                    for sub in _chunks(still_missing, RETRY_BATCH_SIZE_LOCAL):
                        sub_df = _fetch_batch_local(sub)
                        if not sub_df.empty:
                            retried_list.append(sub_df)

                    if retried_list:
                        close_df = pd.concat([close_df] + retried_list, axis=1)
                        close_df = close_df.loc[:, ~close_df.columns.duplicated()]

                    got_cols = set(map(str, close_df.columns))
                    still_missing = [c for c in batch_codes if c not in got_cols]
                    print(f"  [RETRY] pass={p+1}/{RETRY_PASSES_LOCAL} remaining_missing={len(still_missing)}", flush=True)

                if still_missing and len(still_missing) <= RETRY_SINGLE_MAX_LOCAL:
                    print(f"  [RETRY-SINGLE] try singles for remaining={len(still_missing)}", flush=True)
                    single_list = []
                    for c in still_missing:
                        sub_df = _fetch_batch_local([c])
                        if not sub_df.empty:
                            single_list.append(sub_df)
                    if single_list:
                        close_df = pd.concat([close_df] + single_list, axis=1)
                        close_df = close_df.loc[:, ~close_df.columns.duplicated()]
                    got_cols = set(map(str, close_df.columns))
                    still_missing = [c for c in batch_codes if c not in got_cols]
                    print(f"  [RETRY-SINGLE] remaining_missing={len(still_missing)}", flush=True)

                if still_missing:
                    print(f"  [MISS] unresolved missing={len(still_missing)} (kept as NA)", flush=True)

            if not close_df.empty:
                close_df.index = pd.to_datetime(close_df.index).normalize()
                close_df = close_df.loc[(close_df.index >= year_start.normalize()) & (close_df.index <= year_end.normalize())]
                close_df = close_df[~close_df.index.duplicated(keep="last")].sort_index()

                full_index = df_existing.index.union(close_df.index).sort_values()
                df_existing = df_existing.reindex(full_index)

                for c in close_df.columns:
                    if c in df_existing.columns:
                        df_existing.loc[close_df.index, c] = close_df[c].values

                status_df = update_status_after_fetch(status_df, batch_codes, close_df)

            processed += len(batch_codes)

            if (processed % SAVE_EVERY == 0) and (processed < total):
                save_year_csv(out_path, names, df_existing)
                save_status_csv(STATUS_CSV, status_df)
                print(f"  [SAVE] checkpoint saved at {processed}/{total}: {out_path} + status", flush=True)

            dt = time.time() - t0
            got_cols_n = len(close_df.columns) if isinstance(close_df, pd.DataFrame) else 0
            print(f"  [BATCH DONE] {start_i+1}-{end_i}/{total} elapsed={dt:.1f}s got_cols={got_cols_n}", flush=True)

            if SLEEP_PER_BATCH > 0:
                time.sleep(SLEEP_PER_BATCH + random.random() * JITTER_SEC)

        save_year_csv(out_path, names, df_existing)
        save_status_csv(STATUS_CSV, status_df)
        print(f"[OK:{label}] updated: {out_path} (trading-days only) + status", flush=True)

    print(f"[DONE:{label}]", flush=True)

#(4-2)_公開関数（reliable/fast）（★追加：呼び出し口）
def update_kabuka_reliable(
    start_date: str,
    end_date: str,
    include_delisted: bool = True,
    active_only: bool = True,
    warmup_days: int = 120,
):
    """
    (1) 確実取得モード：時間がかかっても欠損を潰す

    active_only=True（デフォルト）:
      - まず直近を取って status を作り
      - last_ok_date がある銘柄（active）だけを対象に過去へ遡る
      - 古い年で「当時存在しない銘柄」への無駄アクセスを大幅に削減
    """
    _update_kabuka_core(
        start_date=start_date,
        end_date=end_date,
        profile=RELIABLE,
        include_delisted=include_delisted,
        label="RELIABLE",
        active_only=active_only,
        warmup_days=warmup_days,
    )

def update_kabuka_fast(start_date: str, end_date: str, include_delisted: bool = False):
    """
    (2) 高速更新モード：直近更新用に速く回す
    """
    _update_kabuka_core(
        start_date=start_date,
        end_date=end_date,
        profile=FAST,
        include_delisted=include_delisted,
        label="FAST",
        active_only=False,
        warmup_days=0,
    )


#(5-1)_最新年から遡って「最終更新日（全銘柄のうち、どれかが値を持つ最終日）」を探す
def detect_last_saved_date_latest_year_first(out_dir: str) -> pd.Timestamp | None:
    """
    kabuka/YYYY.csv を対象に、YYYYが最大のファイルから順に
    「2列目以降に1つでも値がある行」の最終日を探す。
    見つからなければ前年へ戻る。
    """
    paths = glob.glob(os.path.join(out_dir, "*.csv"))
    if not paths:
        return None

    year_paths = []
    for p in paths:
        m = re.search(r"(\d{4})\.csv$", os.path.basename(p))
        if not m:
            continue
        year_paths.append((int(m.group(1)), p))

    if not year_paths:
        return None

    year_paths.sort(key=lambda x: x[0], reverse=True)  # 最新年→過去年

    print(f"[SCAN] kabuka files={len(year_paths)} latest={year_paths[0][0]}", flush=True)

    for year, p in year_paths:
        last_date_str = None
        row_count = 0
        print(f"[OPEN] {year}.csv", flush=True)

        try:
            with open(p, "r", encoding="utf-8-sig", newline="") as f:
                r = csv.reader(f)
                try:
                    next(r)  # code
                    next(r)  # name
                except StopIteration:
                    print(f"  [SKIP] {year}.csv no data rows", flush=True)
                    continue

                for row in r:
                    row_count += 1
                    if not row or not row[0]:
                        continue
                    # 2列目以降に何か入っていれば有効行
                    if any(cell.strip() != "" for cell in row[1:]):
                        last_date_str = row[0].strip()

        except Exception as e:
            print(f"  [ERROR] read failed: {year}.csv {type(e).__name__}: {e}", flush=True)
            continue

        if last_date_str:
            try:
                last_dt = pd.to_datetime(last_date_str)
                print(f"  [FOUND] {year}.csv rows={row_count} last={last_dt.date()}", flush=True)
                return last_dt
            except Exception as e:
                print(f"  [ERROR] date parse failed: {year}.csv last_date_str={last_date_str} err={e}", flush=True)
                continue
        else:
            print(f"  [NONE] {year}.csv rows={row_count} -> go prev year", flush=True)

    return None

#(5-2)_最終日+1日 から 今日（JST）まで差分更新する（★更新：FASTを呼ぶ）
def update_kabuka_from_last_saved(out_dir: str):
    last = detect_last_saved_date_latest_year_first(out_dir)

    today_jst = pd.Timestamp.now(tz="Asia/Tokyo").normalize().tz_localize(None)

    if last is None:
        start_ts = today_jst - pd.Timedelta(days=RECENT_DAYS_IF_EMPTY)
        start_date = start_ts.strftime("%Y-%m-%d")
        end_date = today_jst.strftime("%Y-%m-%d")
        print(f"[AUTO] no existing valid data. update recent {RECENT_DAYS_IF_EMPTY} days: {start_date} - {end_date}", flush=True)
        update_kabuka_fast(start_date, end_date, include_delisted=False)
        return

    start_ts = (pd.to_datetime(last) + pd.Timedelta(days=1)).normalize()
    if start_ts > today_jst:
        print(f"[AUTO] already up-to-date. last_data_day={pd.to_datetime(last).date()} today={today_jst.date()}", flush=True)
        return

    start_date = start_ts.strftime("%Y-%m-%d")
    end_date = today_jst.strftime("%Y-%m-%d")
    print(f"[AUTO] update from last_data_day+1: {start_date} - {end_date} (last_data_day={pd.to_datetime(last).date()})", flush=True)

    # ★自動差分更新はFASTが基本
    update_kabuka_fast(start_date, end_date, include_delisted=False)

#(6)_実行（別セル推奨：このブロックは「関数定義が済んだ後」に実行する）
# 目的：update_kabuka_reliable が未定義のまま呼ばれて落ちるのを防ぐ

if "update_kabuka_reliable" not in globals():
    raise RuntimeError(
        "update_kabuka_reliable が未定義です。\n"
        "→ まず『インポート〜各def定義まで』のセル（この長いコードのセル）を、エラー無しで最後まで実行してください。\n"
        "   その後、この実行セル（#(6)）を実行してください。"
    )

# [1] 確実に取得（取引日だけ保存）
# update_kabuka_reliable("2026-01-01", "2026-01-29")

# [2] 高速更新（取引日だけ保存）
update_kabuka_fast("2026-01-01", "2026-02-06")

[SKIP] invalid code format (not fetched): ['130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '166', '167', '168', '170', '171', '172', '173', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '188', '189', '190', '191', '192', '194', '195', '196', '197', '198', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '215', '216', '217', '218', '219', '220', '221', '222', '223', '224', '226', '227', '228', '229', '233', '234', '235', '236', '237', '238', '239', '241', '243', '244', '245', '246', '247', '248', '249', '250', '251', '252', '253', '254', '255', '256', '257', '258', '261', '262', '263', '264', '265', '266', '267', '268', '269', '271', '273', '274', '276', '277', '278', '279', '280', '281', '282', '283'

In [14]:
# ============================================================
# #(7) 既存の Data/kabuka/YYYY.csv に「N225 列だけ」を後付けで埋める
#  - 既存の他列・既存セルは変更しない（生CSVのまま追記）
#  - すでに N225 列がある場合：空セルだけ埋める（既存値は上書きしない）
#  - yfinance: ^N225 を取得して N225 として保存
# ============================================================

def _fetch_n225_close_map(start_date: str, end_exclusive: str) -> dict[str, float]:
    """
    start_date: 'YYYY-MM-DD'
    end_exclusive: 'YYYY-MM-DD'（排他的）
    戻り: {'YYYY-MM-DD': close_float, ...}
    """
    df = fetch_close_df_yf_batch(
        codes=["N225"],
        start=start_date,
        end=end_exclusive,
        timeout_sec=180.0,
        threads=False,
        suppress_yf_stdout=True,
    )
    if df is None or df.empty or "N225" not in df.columns:
        return {}

    s = df["N225"].dropna()
    out = {}
    for dt, v in s.items():
        dstr = pd.to_datetime(dt).strftime("%Y-%m-%d")
        try:
            out[dstr] = float(v)
        except Exception:
            continue
    return out


def append_n225_to_existing_kabuka_csvs(
    out_dir: str = OUT_DIR,
    year_start: int = 2000,
    year_end: int = 2025,
):
    """
    既存の kabuka/YYYY.csv に対して「N225 列だけ」を追記・補完する。
    - 既存の他列/セルは変更しない
    - N225 列が無ければ末尾に追加
    - N225 列があっても「空欄のみ」埋める（上書きしない）
    """
    for year in range(int(year_start), int(year_end) + 1):
        path = os.path.join(out_dir, f"{year}.csv")
        if not os.path.exists(path):
            print(f"[SKIP] not found: {path}", flush=True)
            continue

        # まず生CSVを読み込む（既存値をそのまま保つため）
        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            rows = list(csv.reader(f))

        if len(rows) < 2:
            print(f"[SKIP] invalid format (<2 rows): {path}", flush=True)
            continue

        code_row = rows[0]
        name_row = rows[1]
        if not code_row or code_row[0] != "code":
            print(f"[SKIP] invalid format (row0 not code): {path}", flush=True)
            continue
        if not name_row or name_row[0] != "name":
            print(f"[SKIP] invalid format (row1 not name): {path}", flush=True)
            continue

        codes = [c.strip() for c in code_row[1:]]
        # N225 の列位置（無ければ追加）
        if "N225" in codes:
            n225_pos = codes.index("N225")  # codes内のindex（0-based）
            is_new_col = False
        else:
            n225_pos = len(codes)
            is_new_col = True
            code_row.append("N225")
            name_row.append("Nikkei 225")
            codes.append("N225")

        # 対象期間：その年の全期間（排他的end）
        start_date = f"{year}-01-01"
        end_exclusive = (pd.Timestamp(year=year, month=12, day=31) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

        n225_map = _fetch_n225_close_map(start_date, end_exclusive)
        if not n225_map:
            print(f"[WARN] N225 fetch empty: year={year} file={path}", flush=True)
            # N225列だけ追加したい場合は続行。埋めないだけ。
        else:
            print(f"[OK] N225 fetched: year={year} points={len(n225_map)}", flush=True)

        # データ行に追記（row = ["YYYY-MM-DD", ...]）
        # 既存セルは変更しない。N225セルが空欄のときだけ埋める。
        updated = 0
        total_rows = 0

        for i in range(2, len(rows)):
            row = rows[i]
            if not row or not row[0]:
                continue
            dstr = row[0].strip()
            if not dstr:
                continue

            total_rows += 1
            v = n225_map.get(dstr, None)

            # その行の列数が足りなければ埋めて長さを合わせる（既存列は空文字で補う）
            need_len = 1 + len(codes)  # date + codes
            if len(row) < need_len:
                row.extend([""] * (need_len - len(row)))

            # N225セルの列index（row側）は 1 + n225_pos
            j = 1 + n225_pos

            # 空欄のみ埋める（上書きしない）
            if row[j].strip() == "" and (v is not None):
                row[j] = str(float(v))
                updated += 1

            rows[i] = row

        # もし N225 が新規列なら、既存行は row長が足りない可能性があるので全行を整形（末尾に空欄追加だけ）
        if is_new_col:
            need_len = 1 + len(codes)
            for i in range(2, len(rows)):
                row = rows[i]
                if not row:
                    continue
                if len(row) < need_len:
                    row.extend([""] * (need_len - len(row)))
                    rows[i] = row

        # 上書き保存（他列は同じ文字列を保持したまま、N225列だけ増える/埋まる）
        with open(path, "w", encoding="utf-8-sig", newline="") as f:
            w = csv.writer(f)
            w.writerows(rows)

        print(f"[DONE] {year}.csv N225 appended={'Y' if is_new_col else 'N'} filled={updated} rows={total_rows}", flush=True)

# 既存の 2000〜2025 に対して「N225列だけ」追記/補完
append_n225_to_existing_kabuka_csvs(OUT_DIR, 2000, 2026)

[OK] N225 fetched: year=2000 points=248
[DONE] 2000.csv N225 appended=N filled=0 rows=260
[OK] N225 fetched: year=2001 points=246
[DONE] 2001.csv N225 appended=N filled=0 rows=261
[OK] N225 fetched: year=2002 points=246
[DONE] 2002.csv N225 appended=N filled=0 rows=261
[OK] N225 fetched: year=2003 points=245
[DONE] 2003.csv N225 appended=N filled=0 rows=261
[OK] N225 fetched: year=2004 points=246
[DONE] 2004.csv N225 appended=N filled=0 rows=262
[OK] N225 fetched: year=2005 points=245
[DONE] 2005.csv N225 appended=N filled=0 rows=260
[OK] N225 fetched: year=2006 points=248
[DONE] 2006.csv N225 appended=N filled=0 rows=260
[OK] N225 fetched: year=2007 points=245
[DONE] 2007.csv N225 appended=N filled=0 rows=245
[OK] N225 fetched: year=2008 points=245
[DONE] 2008.csv N225 appended=N filled=0 rows=245
[OK] N225 fetched: year=2009 points=242
[DONE] 2009.csv N225 appended=N filled=0 rows=243
[OK] N225 fetched: year=2010 points=243
[DONE] 2010.csv N225 appended=N filled=0 rows=243
[OK] N225 